# Drought analysis


In [ ]:
# general
from tqdm import tqdm
from pathlib import Path
import pandas as pd
import yaml 
import os

# geospatial
import geopandas as gpd
import xarray as xr

wd = Path('/home/rooda/OneDrive/Projects/Whatif-Andes')
os.chdir(wd) 

# import config and local module
with open("config.yml") as f:
    cfg = yaml.safe_load(f)

from src import droughts

## Data

In [ ]:
# andeanGC
metadata_andeangc   = pd.read_csv(Path(cfg["dir"]["andeangc"]) / "dataset/AndeanGC_metadata.csv", index_col=0)
runoff_andeangc_obs = pd.read_csv(Path(cfg["dir"]["andeangc"]) / "dataset/AndeanGC_data_1950_2024.csv", index_col=0)
runoff_andeangc_obs.index = pd.to_datetime(runoff_andeangc_obs.index)  # Ensure datetime index
runoff_andeangc_obs = runoff_andeangc_obs.loc[cfg["time"]["ref_start"]:cfg["time"]["ref_end"]]
runoff_andeangc_obs = (runoff_andeangc_obs*1000*86400) / (metadata_andeangc.basin_area*1e6)

runoff_andeangc_sim = xr.open_dataset("data/hydro/runoff_sim_andeangc.nc").q_mm_sim
runoff_andeangc_sim = runoff_andeangc_sim.to_dataframe().reset_index().pivot(columns="gauge_id", index="date")
runoff_andeangc_sim.columns = runoff_andeangc_sim.columns.droplevel(0)
runoff_andeangc_sim = runoff_andeangc_sim.where(runoff_andeangc_obs.notna())
runoff_andeangc_obs = runoff_andeangc_obs[runoff_andeangc_sim.columns]  # align gauges

In [ ]:
# hydroATLAS
metadata_hydroatlas = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg").set_index("gauge_id")
metadata_hydroatlas = metadata_hydroatlas[metadata_hydroatlas.glacier_area_RGI60 > 0] # only the modeled ones
runoff_hydroatlas_sim = xr.open_dataset("data/hydro/runoff_sim_andeangc.nc").q_mm_sim

runoff_hydroatlas_whatif = xr.open_dataset("data/hydro/runoff_sim_hydroatlas_whatif.nc").Q_mm_day
runoff_hydroatlas_whatif = runoff_hydroatlas_whatif.groupby('whatif_scenario').mean() # multi model mean

## Streamflow droughts comparison for AndeanGC

In [ ]:
df = pd.DataFrame({
    'Q85_obs': runoff_andeangc_obs.quantile(cfg["hydro"]["drought_threshold"]),
    'Q85_sim': runoff_andeangc_sim.quantile(cfg["hydro"]["drought_threshold"])})

droughts_andeangc_obs = droughts.find_droughts(runoff_andeangc_obs, cfg["hydro"]["drought_threshold"], threshold_df = runoff_andeangc_obs)
df["duration_obs"] = droughts_andeangc_obs.groupby('gauge_id').duration.sum() 
df["min_value_obs"] = droughts_andeangc_obs.groupby('gauge_id').min_value.mean()
df["severity_obs"] = droughts_andeangc_obs.groupby('gauge_id').severity.sum() 

droughts_andeangc_sim = droughts.find_droughts(runoff_andeangc_sim, cfg["hydro"]["drought_threshold"], threshold_df = runoff_andeangc_obs)
df["duration_sim"] = droughts_andeangc_sim.groupby('gauge_id').duration.sum() 
df["min_value_sim"] = droughts_andeangc_sim.groupby('gauge_id').min_value.mean()
df["severity_sim"] = droughts_andeangc_sim.groupby('gauge_id').severity.sum() 
df.index.name = "gauge_id"
df.to_csv("data/hydro/performance_lowflows.csv")

## Streamflow droughts for all HydroATLAS basins (+ drought extent)

In [ ]:
# Calculate reference droughts
droughts_ref = droughts.find_droughts(runoff_hydroatlas_sim, cfg["hydro"]["drought_threshold"])
droughts_ts = droughts.drought_area_timeseries(droughts_ref, metadata_hydroatlas).rename("historical")

# Initialize results list
df = []

# Process each warming scenario
for sce in tqdm(['15C', '28C', '40C']): 
    # Prepare scenario data
    runoff_whatif_sce = runoff_hydroatlas_whatif.sel(whatif_scenario=sce, drop=True).to_dataframe()
    runoff_whatif_sce = runoff_whatif_sce.reset_index().pivot(columns="gauge_id", index="date")
    runoff_whatif_sce = runoff_whatif_sce.droplevel(level=0, axis=1)

    # Find droughts for scenario
    drought_whatif = droughts.find_droughts(runoff_whatif_sce, cfg["hydro"]["drought_threshold"], runoff_hydroatlas_sim)
    drought_whatif_ts = droughts.drought_area_timeseries(drought_whatif, metadata_hydroatlas).rename(sce)
    droughts_ts = pd.concat([droughts_ts, drought_whatif_ts], axis=1)

    # Calculate changes relative to reference
    duration_change = drought_whatif.groupby('gauge_id').duration.sum() / droughts_ref.groupby('gauge_id').duration.sum()
    min_q_change = drought_whatif.groupby('gauge_id').min_value.mean() / droughts_ref.groupby('gauge_id').min_value.mean()
    severity_change = drought_whatif.groupby('gauge_id').severity.sum() / droughts_ref.groupby('gauge_id').severity.sum()

    # Combine results for this scenario
    scenario_results = pd.concat([duration_change, min_q_change, severity_change], axis=1)
    scenario_results["warming_level"] = sce
    df.append(scenario_results)

# Combine all scenarios and convert to change factors (subtract 1)
df = pd.concat(df)
df[['duration', 'severity', 'min_value']] = (df[['duration', 'severity', 'min_value']] - 1)*100
df.to_csv("data/hydro/droughts_hydroatlas_whatif.csv")
droughts_ts.to_csv("data/hydro/droughts_hydroatlas_area_ts.csv")